# Dynamic Text2SQL FastAPI on Colab

Runs the FastAPI app (Qwen2.5-3B + LoRA adapter, dynamic MySQL/PostgreSQL connection) on a Colab GPU runtime, exposed publicly via ngrok.

**Before running:**
1. Runtime -> Change runtime type -> GPU (T4 or better)
2. Have your ngrok authtoken ready (free at https://dashboard.ngrok.com/get-started/your-authtoken)
3. Have your checkpoint-3480 LoRA adapter available in Google Drive

In [ ]:
!nvidia-smi

## 1. Clone the repo

In [ ]:
REPO_URL = "https://github.com/PrathapShanmugam3/dynamic_text2sql_fastapi.git"

!rm -rf /content/app_repo
!git clone $REPO_URL /content/app_repo
%cd /content/app_repo

## 2. Install dependencies
Colab already has a matching CUDA torch build, so torch is skipped from requirements.txt to avoid a slow reinstall.

In [ ]:
!grep -v -i '^torch' requirements.txt > requirements_colab.txt
!pip install -q -r requirements_colab.txt
!pip install -q pyngrok

## 3. Mount Google Drive (for the LoRA checkpoint)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Configure environment variables
Set `ADAPTER_PATH` to wherever `checkpoint-3480` lives in your Drive.

In [ ]:
import os

os.environ["BASE_MODEL"] = "Qwen/Qwen2.5-3B-Instruct"
os.environ["ADAPTER_PATH"] = "/content/drive/MyDrive/texttosql/results/checkpoint-3480"  # <-- update this path
os.environ["MAX_NEW_TOKENS"] = "128"

assert os.path.exists(os.environ["ADAPTER_PATH"]), "ADAPTER_PATH not found -- fix the path above"

### 4b. Copy checkpoint to local disk (recommended)
Reading many files directly from a mounted Drive can be very slow or hang. Copying once to local Colab disk fixes this.

In [ ]:
import shutil

drive_adapter_path = os.environ["ADAPTER_PATH"]
local_adapter_path = "/content/checkpoint-3480"

if not os.path.exists(local_adapter_path):
    shutil.copytree(drive_adapter_path, local_adapter_path)

os.environ["ADAPTER_PATH"] = local_adapter_path
print("ADAPTER_PATH now points to local disk:", os.environ["ADAPTER_PATH"])

## 5. Configure ngrok and start the tunnel
`.env` is gitignored, so it won't be in the cloned repo. Upload your local `.env` file here (it already has `NGROK_AUTHTOKEN` in it).

In [ ]:
!pip install -q python-dotenv
from google.colab import files
uploaded = files.upload()  # select your local .env file
!mv .env /content/app_repo/.env

In [ ]:
from dotenv import load_dotenv
from pyngrok import ngrok, conf

load_dotenv("/content/app_repo/.env")
NGROK_AUTHTOKEN = os.environ.get("NGROK_AUTHTOKEN")

if not NGROK_AUTHTOKEN:
    import getpass
    NGROK_AUTHTOKEN = getpass.getpass("Enter your ngrok authtoken: ")

conf.get_default().auth_token = NGROK_AUTHTOKEN

## 6. Launch FastAPI (background) + open the tunnel

In [ ]:
import subprocess, sys, time, threading

server_process = subprocess.Popen(
    ["python3", "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content/app_repo",
    env=os.environ.copy(),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

server_ready = threading.Event()
log_lines = []

def _stream_logs():
    for line in server_process.stdout:
        log_lines.append(line)
        print(line, end="")
        if "Application startup complete" in line:
            server_ready.set()
        if server_process.poll() is not None:
            break

threading.Thread(target=_stream_logs, daemon=True).start()

print("Waiting for model to load (streaming logs below)...")
if server_ready.wait(timeout=600):
    print("\nServer is ready.")
else:
    print("\nTimed out after 10 minutes -- check the logs above for what's stuck.")

In [ ]:
tunnel = ngrok.connect(8000, "http")
public_url = tunnel.public_url
print("Public API URL:", public_url)
print("Docs:", public_url + "/docs")
print("Health check:", public_url + "/health")

## 7. Check status (optional)
Logs already stream live in the cell above. Run this only to check whether the server process is still alive.

In [ ]:
exit_code = server_process.poll()
print("Still running" if exit_code is None else f"Exited with code {exit_code}")

## 8. Test the API

In [ ]:
import requests

resp = requests.get(f"{public_url}/health")
print(resp.status_code, resp.json())

In [ ]:
import os

# Sanity check: what DB_* values does THIS notebook process have loaded right now,
# and does the .env file on disk actually contain them?
for k in ["DB_TYPE", "DB_HOST", "DB_PORT", "DB_NAME", "DB_USER", "DB_PASSWORD"]:
    v = os.environ.get(k)
    masked = (v[:4] + "..." + v[-4:]) if v and len(v) > 8 else v
    print(f"{k} = {masked!r} (len={len(v) if v else 0})")

print("\n--- /content/app_repo/.env on disk ---")
with open("/content/app_repo/.env") as f:
    for line in f:
        if line.strip().startswith("DB_"):
            key = line.split("=", 1)[0]
            print(key, "= <present>" if "=" in line else "<malformed line>")


In [ ]:
payload = {
    "question": "Get all employees",
    "allow_limit": 1000
}

resp = requests.post(f"{public_url}/api/ask", json=payload)
print(resp.status_code)
print(resp.json())

## 9. Shutdown (run when done)

In [ ]:
ngrok.disconnect(public_url)
server_process.terminate()